## Entendendo a Estratégia LSTM do Projeto 🧠

Olá! Este notebook vai te guiar, passo a passo, por como a Rede Neural Recorrente (RNN) do tipo LSTM (Long Short-Term Memory) está sendo usada no seu projeto para analisar o mercado financeiro.

Vamos usar analogias do dia-a-dia para entender os métodos mais importantes que vimos no arquivo `src/strategies/lstm.py`.

### 1. O que é uma RNN e por que LSTM?

**Analogia Rápida:** Pense em ler um livro. Você não entende uma palavra apenas olhando para ela isoladamente; você a entende com base nas palavras que vieram *antes* dela. Seu cérebro "lembra" o começo da frase para dar contexto ao final.

Uma **RNN (Rede Neural Recorrente)** funciona assim: ela tem uma "memória" (um estado interno) que permite que informações de etapas anteriores influenciem a etapa atual. Isso é perfeito para o mercado financeiro, onde o preço de hoje é influenciado pelos preços de ontem, anteontem, etc.

A **LSTM (Long Short-Term Memory)** é um tipo *avançado* de RNN. Ela é como um leitor com uma memória superpotente. Ela não só lembra do passado recente (memória de curto prazo), mas também tem "portões" especiais que permitem decidir o que é importante guardar para o futuro (memória de longo prazo) e o que pode ser esquecido. Ela é ótima em aprender padrões que ocorreram há, digamos, 50 ou 100 barras de preço no passado.

### 2. Definindo as "Pistas" (Features)

Antes de pedirmos para a LSTM prever o futuro, precisamos dizer a ela *no que* ela deve prestar atenção. Não podemos simplesmente jogar os preços brutos para ela.

**Analogia:** Imagine que você é um detetive tentando prever se vai chover. Você não olha *apenas* se o céu está nublado agora. Você coleta várias "pistas" (features): a umidade do ar, a velocidade do vento, a temperatura, se choveu nos últimos 3 dias, etc.

No nosso caso, o método `define_features` faz exatamente isso: ele cria "pistas" financeiras (indicadores técnicos) a partir dos dados brutos do preço.

No seu código (`src/strategies/lstm.py`), a classe `LSTMStrategy` define estas pistas:

In [ ]:
# Este é um trecho da sua classe LSTMStrategy (do arquivo lstm.py)

def define_features(self, data: pd.DataFrame) -> pd.DataFrame:
    """
    Adiciona os indicadores técnicos que servirão de features para o modelo.
    """
    df = data.copy()
    
    # Pistas sobre a tendência (Médias Móveis)
    df['ema_9'] = df['close'].ewm(span=9, adjust=False).mean()
    df['sma_20'] = df['close'].rolling(window=20).mean()
    df['sma_50'] = df['close'].rolling(window=50).mean()
    df['sma_200'] = df['close'].rolling(window=200).mean()

    # Pista sobre o "entusiasmo" do mercado (Volume)
    if 'volume' not in df.columns:
        df['volume'] = 0

    # Pista sobre a "agitação" do mercado (Volatilidade)
    df['returns'] = df['close'].pct_change()
    volatility_window = min(21, len(df) - 1) if len(df) > 1 else 1
    if volatility_window > 0:
         df['volatility'] = df['returns'].rolling(window=volatility_window).std()
    else:
         df['volatility'] = 0.0

    # Pista sobre "cansaço" do movimento (RSI - Índice de Força Relativa)
    rsi_period = 14
    delta = df['close'].diff()
    gain = delta.where(delta > 0, 0).rolling(window=rsi_period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=rsi_period).mean()
    rs = gain / loss.replace(0, 1e-6) 
    df['rsi'] = 100 - (100 / (1 + rs))
    df['rsi'] = df['rsi'].fillna(50) 

    # ... (limpeza de dados) ...
    return df

### 3. Definindo o "Alvo" (Target)

Agora que temos as pistas, precisamos dizer à LSTM o que ela deve acertar. O que queremos prever?

**Analogia:** Se as "pistas" são umidade, vento e temperatura, o "alvo" (target) é a resposta para a pergunta: "Vai chover amanhã?" (Sim ou Não).

No seu projeto, o alvo é definido pela função `calculate_target` (que está no arquivo `src/strategies/base.py`). Ela faz uma pergunta simples: o preço de fechamento daqui a `target_period` (ex: 1 barra) vai ser *maior* que o preço de fechamento de agora?

- Se **Sim**, o alvo é `1` (sinal de "Sobe" ou "Compra").
- Se **Não** (for igual ou menor), o alvo é `0` (sinal de "Desce" ou "Vende").

Estamos transformando o problema de "adivinhar o preço" (que é muito difícil) em um problema de "adivinhar a direção" (mais fácil, Sim/Não).

### 4. Normalizando os Dados (MinMaxScaler)

Temos pistas com escalas muito diferentes. O `volume` pode estar na casa dos milhões, enquanto o `rsi` vai de 0 a 100 e a `volatilidade` pode ser 0.001.

**Analogia:** Imagine tentar cozinhar uma receita onde um ingrediente está em *toneladas* (volume) e outro em *miligramas* (RSI). A rede neural vai achar que o ingrediente em toneladas é *infinitamente* mais importante, só por causa da escala. 

Para consertar isso, nós "normalizamos" todas as pistas. Usamos o `MinMaxScaler` para colocar todas elas na mesma escala, geralmente entre 0 e 1. Assim, a rede pode comparar `volume`, `rsi` e `volatilidade` de forma justa. 

Isso acontece dentro da classe `LSTMWrapper`:

In [ ]:
# Trecho de LSTMWrapper (do arquivo lstm.py)

class LSTMWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, ...):
        # ...
        # Aqui criamos o "conversor" de escalas
        self.scaler = MinMaxScaler(feature_range=(0, 1))
        
    def fit(self, X, y):
        # ...
        # Aqui nós "aprendemos" a escala (fit) e "convertemos" os dados (transform)
        X_scaled = self.scaler.fit_transform(X_values)
        # ...
        
    def predict(self, X):
        # ...
        # Na hora de prever, usamos a *mesma* conversão que aprendemos no treino
        X_scaled = self.scaler.transform(X_values)
        # ...

### 5. Criando as "Janelas" de Memória (create_sequences)

Este é o **coração** do processo da LSTM. A rede não olha para as pistas de um dia (uma barra) e tenta prever a próxima. Ela olha para uma *sequência* de dias passados para prever o dia seguinte.

O parâmetro `lookback` (ex: 60) define o tamanho dessa "janela" de memória.

**Analogia:** Para prever a próxima palavra na frase "O rápido cachorro marrom...", você não olha só para a palavra "marrom". Você olha para a sequência inteira (`lookback` = 4). A função `create_sequences` faz isso: ela desliza uma "janela" pelos dados.

1.  **Amostra 1:** Pega as pistas das barras [1 a 60] -> Tenta prever o alvo da barra [61]
2.  **Amostra 2:** Pega as pistas das barras [2 a 61] -> Tenta prever o alvo da barra [62]
3.  **Amostra 3:** Pega as pistas das barras [3 a 62] -> Tenta prever o alvo da barra [63]

...e assim por diante. Estamos criando um conjunto de dados onde cada "X" é uma sequência de 60 passos e cada "y" é a resposta do passo seguinte.

In [ ]:
# Esta é a função auxiliar (do arquivo lstm.py)

def create_sequences(X_data, y_data, lookback):
    """
    Transforma um array de features e um array de targets em sequências
    para alimentar a LSTM.
    """
    X, y = [], []
    for i in range(len(X_data) - lookback):
        # X é uma sequência de 'lookback' passos (ex: 60 dias de pistas)
        X.append(X_data[i:(i + lookback), :])
        # y é o alvo *após* essa sequência (ex: o alvo do dia 61)
        y.append(y_data[i + lookback])
    return np.array(X), np.array(y)

# E é usada dentro do 'fit' do LSTMWrapper:
# X_seq, y_seq = create_sequences(X_scaled, y_values, self.lookback)

### 6. Construindo o "Cérebro" (\_build\_model)

Agora, montamos a arquitetura da rede neural. É como empilhar blocos de Lego para construir um cérebro.

**Analogia (os blocos):**

* `LSTM(units=50, ...)`: Esta é a camada de **memória principal**. Ela recebe a sequência (a "janela" de 60 passos) e processa tudo, decidindo o que lembrar e o que esquecer. O `units=50` é como dizer que essa camada tem 50 "neurônios" de memória trabalhando em conjunto.
* `Dropout(0.2)`: Esta é uma camada de **"prevenção de vícios"** (regularização). Durante o treino, ela desliga aleatoriamente 20% dos neurônios. Isso força a rede a não depender de um único "neurônio-estrela" para tomar decisões, tornando-a mais robusta. É como treinar um time de basquete forçando seu melhor jogador a ficar no banco de vez em quando, para que o resto do time aprenda a pontuar.
* `Dense(units=1, activation='sigmoid')`: Esta é a camada de **decisão final**. Após todas as camadas de memória processarem a informação, esta camada pega o resultado final e o "esmaga" em um único número entre 0 e 1 (por causa da ativação `sigmoid`). Esse número é a *probabilidade* do alvo ser 1. (ex: 0.80 = "Estou 80% confiante de que o preço vai subir").

In [ ]:
# Trecho de LSTMWrapper (do arquivo lstm.py)

def _build_model(self):
    """Define a arquitetura da rede LSTM."""
    model = Sequential()
    
    # Camada de memória que recebe a sequência (shape: lookback, n_features)
    model.add(LSTM(units=self.lstm_units, return_sequences=True, input_shape=(self.lookback, self.n_features)))
    model.add(Dropout(0.2)) # Regularização
    
    # Outra camada de memória
    model.add(LSTM(units=self.lstm_units, return_sequences=False))
    model.add(Dropout(0.2)) # Regularização
    
    model.add(Dense(units=25)) # Camada de processamento intermediário
    
    # Camada final de decisão (0 ou 1)
    model.add(Dense(units=1, activation='sigmoid')) 
    
    # Compila o modelo, definindo como ele vai aprender
    # loss: como medir o erro (binary_crossentropy é para Sim/Não)
    # optimizer: como ajustar os pesos para diminuir o erro (adam é um otimizador popular)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

### 7. O Treinamento (fit)

Aqui é onde a mágica acontece. O método `fit` é o processo de "estudo" da rede.

**Analogia:** É como fazer uma prova com gabarito. 
1.  A rede pega uma "janela" de dados (`X_seq`).
2.  Ela faz uma previsão (ex: "Acho que é 1 - Sobe").
3.  Ela compara a previsão com o gabarito (`y_seq`, ex: "Era 0 - Desce").
4.  Ela calcula o quão feio errou (o `loss`).
5.  Ela usa o `optimizer` para ajustar *milhões* de pequenos "botões" (pesos) internos em suas camadas LSTM e Dense, para que da *próxima vez* que vir uma sequência parecida, ela erre menos.

Ela repete isso milhares de vezes (`batch_size`) por várias "rodadas" completas pelos dados (`epochs`).

In [ ]:
# Trecho do 'fit' do LSTMWrapper (do arquivo lstm.py)

def fit(self, X, y):
    # ... (preparação dos dados, scaling, criação de sequências) ...
    
    # X_seq e y_seq estão prontos
    
    # ... (lógica de 'early stopping' para parar de treinar se não houver melhora)

    if validation_split > 0:
        # Inicia o processo de "estudo"
        self.history = self.model.fit(
            X_seq, y_seq,
            epochs=self.epochs,
            batch_size=self.batch_size,
            validation_split=validation_split,
            callbacks=[early_stopping],
            verbose=0 
        )
    # ... (lógica para treinar sem validação)
    
    return self

### 8. A Previsão (predict)

Depois que o modelo está treinado (estudou e "passou na prova"), ele está pronto para o trabalho.

**Analogia:** Agora é a prova *real*, sem gabarito. 

Nós pegamos os dados mais recentes (as últimas 60 barras de "pistas"), aplicamos a *mesma* normalização (`scaler.transform`), criamos a *mesma* estrutura de sequência e entregamos ao `model.predict()`. 

O modelo não sabe o que vai acontecer. Ele apenas aplica o conhecimento que aprendeu no treino e nos dá sua melhor probabilidade (ex: 0.80). Nós então arredondamos isso: se for > 0.5, consideramos `1` (Sobe), senão `0` (Desce).

In [ ]:
# Trecho do 'predict' do LSTMWrapper (do arquivo lstm.py)

def predict(self, X):
    # ... (preparação e scaling de X)
    X_scaled = self.scaler.transform(X_values)
    
    # Cria as sequências (não precisamos de 'y' aqui)
    y_dummy = np.zeros(len(X_scaled))
    X_seq, _ = create_sequences(X_scaled, y_dummy, self.lookback)
    
    if len(X_seq) == 0:
        return np.array([], dtype=int)
        
    # 1. Modelo prevê a probabilidade (ex: 0.80)
    predictions_proba = self.model.predict(X_seq)
    
    # 2. Convertemos a probabilidade em decisão (ex: 0.80 > 0.5 -> 1)
    predictions = (predictions_proba > 0.5).astype(int)
    
    return predictions.flatten()

### 9. Salvando o Cérebro (save / load)

Treinar uma rede neural pode levar horas. Não queremos repetir isso toda vez que ligamos o robô. Por isso, o `train_model.py` (que você também anexou) chama o método `save`.

**Analogia:** Depois de estudar o semestre inteiro e se formar, você não joga seu cérebro fora! Você o "salva" para usar no mercado de trabalho. 

O método `save` da `LSTMStrategy` (que por sua vez chama o `save` do `LSTMWrapper`) salva três coisas importantes:

1.  **O Modelo Keras (`_lstm.keras`):** O "cérebro" treinado, com todos os seus milhões de pesos ajustados.
2.  **O Scaler (`_scaler.joblib`):** O "conversor de unidades" que aprendemos no treino. É *crucial* usar o mesmo conversor nos dados novos.
3.  **Os Parâmetros (`_params.joblib`):** Informações vitais, como o `lookback`, para sabermos como reconstruir o modelo da forma correta.

O método `load` faz o caminho inverso: ele reconstrói o `LSTMWrapper` a partir desses arquivos, deixando-o pronto para fazer previsões.

In [ ]:
# Métodos de persistência do LSTMWrapper (do arquivo lstm.py)

def save(self, model_path_prefix: str):
    # ...
    model_path = f"{model_path_prefix}_lstm.keras"
    scaler_path = f"{model_path_prefix}_scaler.joblib"
    params_path = f"{model_path_prefix}_params.joblib"
    
    self.model.save(model_path)
    joblib.dump(self.scaler, scaler_path)
    params_to_save = {'lookback': self.lookback, 'n_features': self.n_features}
    joblib.dump(params_to_save, params_path)
    # ...

@classmethod
def load(cls, model_path_prefix: str):
    # ...
    loaded_keras_model = keras.models.load_model(model_path)
    loaded_scaler = joblib.load(scaler_path)
    loaded_params = joblib.load(params_path)
    # ...
    # Cria uma nova instância com os componentes carregados
    instance = cls(
        lookback=loaded_params.get('lookback', 60), 
        n_features=loaded_params.get('n_features', 1) 
    )
    instance.model = loaded_keras_model
    instance.scaler = loaded_scaler
    return instance

## Resumo do Fluxo

1.  **`train_model.py` é executado.**
2.  Ele carrega a `LSTMStrategy`.
3.  Pede à estratégia para criar **Pistas** (`define_features`).
4.  Pede à estratégia para criar o **Alvo** (`define_target`).
5.  Pede à estratégia um modelo (`define_model`), que é o `LSTMWrapper`.
6.  Chama o `fit` do `LSTMWrapper`:
    * Dados são **Normalizados** (scaled).
    * Dados são transformados em **Sequências** (janelas de `lookback`).
    * O **Cérebro** (`_build_model`) é treinado (`model.fit`) usando essas sequências.
7.  `train_model.py` chama o `save` da estratégia, que salva o cérebro, o normalizador e os parâmetros no disco.

Espero que isso tenha tornado o processo mais claro! É uma arquitetura muito poderosa que você está construindo.